# N-Grams Feature Generation

In [ ]:
import sys
sys.path.append("..")
import glob
import os
from helpers.data_prep_and_print import print_df
from pyspark.sql.functions import col, last, lit, posexplode, regexp_extract, split, when
from pyspark.ml.feature import NGram, Tokenizer
from pyspark.sql import SparkSession, Window

In [ ]:
# Create local StreamingContext with a batch interval of 10s
spark = (SparkSession.builder
           .appName("N-Gram Creation")
           .getOrCreate())

## Read in the Data Files

In [ ]:
# https://archive.ics.uci.edu/dataset/311/sentence+classification 
input_folder = "../data/labeled_articles/"
file_list = sorted(glob.glob(os.path.join(input_folder, "*.txt")))
dfs = []
for filename in file_list:
    lines = (spark.read.option("wholetext", "true").text(filename)
             .select(posexplode(split(col("value"), r"\r?\n")).alias("line_number", "value")))
    section_marker = when(
        col("value").startswith("###"),
        regexp_extract(col("value"), r"^###\s*(.*?)\s*###$", 1),
    )
    section_window = Window.orderBy("line_number").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    temp_df = (lines.withColumn("section_marker", section_marker)
               .withColumn("section", last("section_marker", ignorenulls=True).over(section_window))
               .select(
                   regexp_extract(col("value"), r"^(\S+)\s+(.*)$", 1).alias("label"),
                   regexp_extract(col("value"), r"^\S+\s+(.*)$", 1).alias("sentence"),
                   "section",
               ))
    temp_df = temp_df.withColumn("filename", lit(os.path.basename(filename)))
    dfs.append(temp_df)
union_df = dfs[0]
for df in dfs[1:]:
    union_df = union_df.union(df)

union_df = union_df.where((col("label") != "") & ~col("label").startswith("###"))
print(union_df.count())
union_df.printSchema()
print_df(union_df, 5)

In [ ]:
# read the labeled articles from labeled_articles.csv and check for null values in the sentence column
#union_df = spark.read.csv("../data/labeled_articles.csv", header=True, inferSchema=True)

## Build Tokens

In [ ]:
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
tokenized_df = tokenizer.transform(union_df)

print_df(tokenized_df.select("sentence", "words"]), 5)

## Build N-Grams

In [ ]:
ngram = NGram(n=2, inputCol="words", outputCol="ngrams")
ngram_df = ngram.transform(tokenized_df)
print_df(ngram_df, 5)

In [ ]:
spark.stop()